# Live Signal Scanner — Google Colab Live Monitor V6

V6 is matched to the Advanced Fast Scanner package.

It supports:
- `channels` fast health scan
- `channels-discovery` deeper TV discovery
- `movies` movie scan with TV→Movie routing repair
- `movies-discovery` deeper movie discovery
- `events`, `today`, `upcoming`
- `all` and `full-audit`

The live monitor displays both overlapping stages:
- Global verification progress
- BD/proxy pipeline progress
- Candidate pool, adaptive first wave, later expansion
- Source cache / output file updates
- Final Git commit and push


In [ ]:
#@title 1) Repository setup — safe authentication
REPO_OWNER = "DigeeGlamour" #@param {type:"string"}
REPO_NAME = "click-tv" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
USE_DRIVE_CACHE = False #@param {type:"boolean"}

import getpass
import json
import os
import shutil
import stat
import subprocess
import tempfile
import urllib.error
import urllib.request
from pathlib import Path

try:
    from google.colab import userdata
except Exception:
    userdata = None


def get_secret(name: str, required: bool = False) -> str:
    value = ""
    if userdata is not None:
        try:
            value = userdata.get(name) or ""
        except Exception:
            value = ""

    if required and not value:
        value = getpass.getpass(f"Enter {name}: ").strip()

    return value.strip()


GITHUB_TOKEN = get_secret("GITHUB_TOKEN", required=True)
TELEGRAM_BOT_TOKEN = get_secret("TELEGRAM_BOT_TOKEN")
TELEGRAM_CHAT_ID = get_secret("TELEGRAM_CHAT_ID")

if not GITHUB_TOKEN:
    raise RuntimeError(
        "GITHUB_TOKEN পাওয়া যায়নি। Colab Secrets-এ GITHUB_TOKEN যোগ করে "
        "Notebook access ON করুন।"
    )

if any(character.isspace() for character in GITHUB_TOKEN):
    raise RuntimeError(
        "GITHUB_TOKEN-এর Value-তে space/newline আছে। শুধু সম্পূর্ণ token paste করুন।"
    )

if GITHUB_TOKEN.count("github_pat_") > 1:
    raise RuntimeError(
        "Token value ভুল: github_pat_ prefix একাধিকবার আছে। "
        "সম্পূর্ণ token একবারই paste করুন।"
    )

if not (
    GITHUB_TOKEN.startswith("github_pat_")
    or GITHUB_TOKEN.startswith("ghp_")
):
    raise RuntimeError(
        "GITHUB_TOKEN সঠিক token format নয়। GitHub-এর আসল PAT paste করুন।"
    )


def validate_github_token(token: str) -> str:
    request = urllib.request.Request(
        "https://api.github.com/user",
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "User-Agent": "Live-Signal-Colab-Scanner",
        },
    )

    try:
        with urllib.request.urlopen(request, timeout=20) as response:
            payload = json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as error:
        if error.code == 401:
            raise RuntimeError(
                "GitHub token invalid বা expired। Token regenerate করে "
                "Colab Secret GITHUB_TOKEN-এর Value বদলান।"
            ) from error
        raise RuntimeError(
            f"GitHub token validation failed with HTTP {error.code}."
        ) from error
    except Exception as error:
        raise RuntimeError(
            f"GitHub token validation request failed: {error}"
        ) from error

    login = str(payload.get("login") or "").strip()
    if not login:
        raise RuntimeError("GitHub token valid user account ফেরত দেয়নি।")

    return login


authenticated_login = validate_github_token(GITHUB_TOKEN)
print(f"✅ GitHub token verified for account: {authenticated_login}")

if USE_DRIVE_CACHE:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/LiveSignalScanner")
else:
    BASE_DIR = Path("/content")

REPO_DIR = BASE_DIR / REPO_NAME
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

# Token command/traceback-এ না দেখিয়ে Git authentication করা হবে।
ASKPASS_PATH = Path(tempfile.gettempdir()) / "live_signal_git_askpass.sh"
ASKPASS_PATH.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' 'x-access-token' ;;
  *Password*) printf '%s\\n' "$GITHUB_TOKEN" ;;
  *) printf '%s\\n' '' ;;
esac
""",
    encoding="utf-8",
)
ASKPASS_PATH.chmod(
    ASKPASS_PATH.stat().st_mode
    | stat.S_IXUSR
    | stat.S_IXGRP
    | stat.S_IXOTH
)

GIT_ENV = os.environ.copy()
GIT_ENV["GIT_ASKPASS"] = str(ASKPASS_PATH)
GIT_ENV["GIT_TERMINAL_PROMPT"] = "0"
GIT_ENV["GITHUB_TOKEN"] = GITHUB_TOKEN


def run_git(args, cwd=None, check=True):
    command = ["git"] + list(args)
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        env=GIT_ENV,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if result.stdout.strip():
        print(result.stdout.rstrip())

    if result.returncode != 0:
        safe_error = result.stderr.replace(GITHUB_TOKEN, "[REDACTED]")
        if safe_error.strip():
            print(safe_error.rstrip())

        if check:
            raise RuntimeError(
                f"Git command failed with exit code {result.returncode}: "
                f"git {' '.join(args)}"
            )

    return result


BASE_DIR.mkdir(parents=True, exist_ok=True)

# Every new notebook setup starts from a clean GitHub checkout.
# This prevents old working files, cache folders, or failed local commits
# from contaminating the next scan.
if REPO_DIR.exists():
    print("[CLEAN START] Removing previous Colab checkout...")
    shutil.rmtree(REPO_DIR)

print("Cloning a fresh repository...")
run_git(
    [
        "clone",
        "--branch",
        BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR),
    ]
)

run_git(["config", "user.name", "Live Signal Colab"], cwd=REPO_DIR)
run_git(
    ["config", "user.email", "colab-scanner@users.noreply.github.com"],
    cwd=REPO_DIR,
)

commit_id = run_git(
    ["rev-parse", "--short", "HEAD"],
    cwd=REPO_DIR,
).stdout.strip()

print("=" * 68)
print(f"✅ Repository ready: {REPO_DIR}")
print(f"   Repository: {REPO_OWNER}/{REPO_NAME}")
print(f"   Branch: {BRANCH}")
print(f"   Commit: {commit_id}")
print("=" * 68)


In [ ]:
#@title 2) Select mode and run with detailed live monitor
SCAN_MODE = "channels" #@param ["channels", "channels-discovery", "events", "today", "upcoming", "movies", "movies-discovery", "all", "full-audit"]
PUSH_RESULTS_TO_GITHUB = True #@param {type:"boolean"}
PROGRESS_EVERY = 25 #@param {type:"integer"}
HEARTBEAT_SECONDS = 20 #@param {type:"integer"}
SHOW_CHANGED_FILES = True #@param {type:"boolean"}

import json
import os
import subprocess
import sys
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

SUPPORTED_MODES = {
    "all",
    "full-audit",
    "channels",
    "channels-discovery",
    "movies",
    "movies-discovery",
    "events",
    "today",
    "upcoming",
}

if SCAN_MODE not in SUPPORTED_MODES:
    raise ValueError(f"Unsupported scan mode: {SCAN_MODE}")

if PROGRESS_EVERY < 1:
    PROGRESS_EVERY = 25

if HEARTBEAT_SECONDS < 10:
    HEARTBEAT_SECONDS = 10

required_paths = [
    REPO_DIR / "scan.py",
    REPO_DIR / "config" / "settings.json",
    REPO_DIR / "config" / "sources.json",
    REPO_DIR / "scanner" / "content_router.py",
    REPO_DIR / "scanner" / "fast_pipeline.py",
    REPO_DIR / "scanner" / "source_loader.py",
    REPO_DIR / "scanner" / "normalizer.py",
    REPO_DIR / "scanner" / "planner.py",
    REPO_DIR / "scanner" / "verifier.py",
    REPO_DIR / "scanner" / "bd_verifier.py",
    REPO_DIR / "scanner" / "channels.py",
    REPO_DIR / "scanner" / "movies.py",
    REPO_DIR / "scanner" / "events.py",
    REPO_DIR / "scanner" / "output.py",
]

missing_paths = [
    str(path.relative_to(REPO_DIR))
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required files missing:\n- " + "\n- ".join(missing_paths)
    )

# Do not silently destroy a completed scan commit that failed only at push.
ahead_result = subprocess.run(
    ["git", "rev-list", "--count", f"origin/{BRANCH}..HEAD"],
    cwd=REPO_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=True,
)
ahead_count = int((ahead_result.stdout or "0").strip() or "0")

if ahead_count > 0:
    raise RuntimeError(
        f"{ahead_count} unpushed local commit পাওয়া গেছে। "
        "Scan আবার চালাবেন না। নিচের '3) Retry last GitHub push only' cell চালান।"
    )

# Always use the latest clean GitHub code.
print("\n[PREPARE] Updating repository from GitHub...")
run_git(["fetch", "origin", BRANCH], cwd=REPO_DIR)
run_git(["checkout", BRANCH], cwd=REPO_DIR)
run_git(["reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)

commit_id = run_git(
    ["rev-parse", "--short", "HEAD"],
    cwd=REPO_DIR,
).stdout.strip()

print("\n[PREFLIGHT] Checking Python syntax and required functions...")
compile_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "py_compile",
        "scan.py",
        "scanner/content_router.py",
        "scanner/fast_pipeline.py",
        "scanner/source_loader.py",
        "scanner/normalizer.py",
        "scanner/planner.py",
        "scanner/verifier.py",
        "scanner/bd_verifier.py",
        "scanner/channels.py",
        "scanner/movies.py",
        "scanner/events.py",
        "scanner/output.py",
    ],
    cwd=REPO_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
if compile_result.stdout.strip():
    print(compile_result.stdout.rstrip())
if compile_result.returncode != 0:
    raise RuntimeError("Python syntax validation failed.")

import_result = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "from scanner.content_router import route_candidate; "
            "from scanner.fast_pipeline import run_fast_verification_pipeline; "
            "from scanner.planner import plan_candidates; "
            "from scanner.channels import process_tv_channels; "
            "from scanner.movies import process_movies; "
            "from scanner.events import process_events; "
            "from scanner.output import publish_scan_outputs; "
            "print('Required imports OK')"
        ),
    ],
    cwd=REPO_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(import_result.stdout.rstrip())
if import_result.returncode != 0:
    raise RuntimeError("Required scanner function import failed.")

settings_path = REPO_DIR / "config" / "settings.json"
settings_original_text = settings_path.read_text(encoding="utf-8")
settings_data = json.loads(settings_original_text)

verification_settings = settings_data.setdefault("verification", {})
original_progress_interval = verification_settings.get(
    "progress_interval",
    100,
)
verification_settings["progress_interval"] = int(PROGRESS_EVERY)
settings_path.write_text(
    json.dumps(settings_data, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

sources_data = json.loads(
    (REPO_DIR / "config" / "sources.json").read_text(encoding="utf-8")
)

def count_configured_sources(data):
    if isinstance(data, list):
        return len(data)
    if not isinstance(data, dict):
        return 0

    count = 0
    for value in data.values():
        if isinstance(value, list):
            count += len(value)
        elif isinstance(value, dict):
            nested_lists = [
                nested_value
                for nested_value in value.values()
                if isinstance(nested_value, list)
            ]
            if nested_lists:
                count += sum(len(item) for item in nested_lists)
    return count

configured_source_count = count_configured_sources(sources_data)

print("\n" + "=" * 72)
print("LIVE SIGNAL COLAB SCANNER — ADVANCED PIPELINE MONITOR")
print(f"Mode                  : {SCAN_MODE}")
print(f"Repository            : {REPO_OWNER}/{REPO_NAME}")
print(f"Branch / Commit       : {BRANCH} / {commit_id}")
print(f"Configured sources    : {configured_source_count}")
print(f"Verification workers  : {verification_settings.get('workers', settings_data.get('verification_workers'))}")
print(f"Stream timeout        : {verification_settings.get('timeout_seconds', settings_data.get('stream_timeout_seconds'))} seconds")
print(f"Progress output every : {PROGRESS_EVERY} completed candidates")
print(f"Heartbeat every       : {HEARTBEAT_SECONDS} seconds")
print(f"Push results          : {PUSH_RESULTS_TO_GITHUB}")
print("=" * 72)

scan_environment = os.environ.copy()
scan_environment["PYTHONUNBUFFERED"] = "1"
scan_environment["PYTHONFAULTHANDLER"] = "1"

if TELEGRAM_BOT_TOKEN:
    scan_environment["TELEGRAM_BOT_TOKEN"] = TELEGRAM_BOT_TOKEN
if TELEGRAM_CHAT_ID:
    scan_environment["TELEGRAM_CHAT_ID"] = TELEGRAM_CHAT_ID

tracked_paths = {
    "candidates": REPO_DIR / "working" / "candidates.json",
    "global-results": REPO_DIR / "working" / "global-results.json",
    "bd-results": REPO_DIR / "working" / "bd-results.json",
    "manifest": REPO_DIR / "data" / "manifest.json",
    "scan-summary": REPO_DIR / "reports" / "scan-summary.json",
    "pipeline-performance": REPO_DIR / "reports" / "pipeline-performance.json",
    "routing-plan": REPO_DIR / "reports" / "preverification-plan.json",
}

baseline_mtimes = {
    name: path.stat().st_mtime if path.exists() else 0
    for name, path in tracked_paths.items()
}

state_lock = threading.Lock()
live_state = {
    "last_line": "Scanner process starting...",
    "stage": "Starting",
    "processed": "",
}
stop_monitor = threading.Event()
started_monotonic = time.monotonic()
started_epoch = time.time()
log_path = Path("/content/live-signal-scanner.log")


def safe_json(path):
    try:
        with path.open("r", encoding="utf-8") as handle:
            return json.load(handle)
    except Exception:
        return {}


def describe_updated_file(name, path):
    if not path.exists():
        return ""

    modified = path.stat().st_mtime
    if modified < started_epoch - 2:
        return ""

    data = safe_json(path)

    if name == "candidates":
        return (
            f"candidates raw={data.get('raw_candidate_count', '?')}, "
            f"normalized={data.get('normalized_candidate_count', '?')}, "
            f"planned={data.get('planned_candidate_count', data.get('total_candidates', '?'))}"
        )

    results = data.get("results")
    if isinstance(results, list):
        return f"{name} results={len(results)}"

    if name == "manifest":
        return "manifest updated"

    if name == "scan-summary":
        return f"scan-summary status={data.get('status', '?')}"

    if name == "pipeline-performance":
        return (
            f"pipeline global={data.get('global_network_checked', '?')}, "
            f"bd={data.get('bd_proxy_submitted', '?')}, "
            f"publishable={data.get('final_publishable', '?')}"
        )

    if name == "routing-plan":
        return (
            f"routing pool={data.get('candidate_pool_count', data.get('planned_candidates', '?'))}, "
            f"first-wave={data.get('initial_wave_candidates', '?')}, "
            f"rerouted={data.get('rerouted_counts', {})}"
        )

    return f"{name} updated"


def monitor_worker(process):
    while not stop_monitor.wait(HEARTBEAT_SECONDS):
        elapsed = int(time.monotonic() - started_monotonic)
        minutes, seconds = divmod(elapsed, 60)

        with state_lock:
            stage = live_state["stage"]
            processed = live_state["processed"]
            last_line = live_state["last_line"]

        file_updates = []
        for name, path in tracked_paths.items():
            description = describe_updated_file(name, path)
            if description:
                file_updates.append(description)

        print(
            f"\n[MONITOR {minutes:02d}:{seconds:02d}] "
            f"alive={process.poll() is None} | stage={stage}"
            + (f" | progress={processed}" if processed else "")
        )
        print(f"   Last activity: {last_line[:220]}")
        if file_updates:
            print("   Updated files: " + " | ".join(file_updates))
        else:
            print("   Updated files: waiting for next atomic output...")
        sys.stdout.flush()


process = subprocess.Popen(
    [sys.executable, "-u", "scan.py", SCAN_MODE],
    cwd=REPO_DIR,
    env=scan_environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

monitor_thread = threading.Thread(
    target=monitor_worker,
    args=(process,),
    daemon=True,
)
monitor_thread.start()

return_code = None

try:
    with log_path.open("w", encoding="utf-8") as log_handle:
        assert process.stdout is not None

        for raw_line in process.stdout:
            line = raw_line.rstrip("\n")
            timestamp = datetime.now().strftime("%H:%M:%S")
            rendered = f"[{timestamp}] {line}"

            print(rendered, flush=True)
            log_handle.write(rendered + "\n")
            log_handle.flush()

            with state_lock:
                if line.strip():
                    live_state["last_line"] = line.strip()

                if "[Step " in line:
                    live_state["stage"] = line.strip()

                if "Verification progress:" in line:
                    live_state["stage"] = "Global stream verification"
                    live_state["processed"] = (
                        line.split("Verification progress:", 1)[1].strip()
                    )

                if "Global progress:" in line:
                    live_state["stage"] = "Adaptive Global verification"
                    live_state["processed"] = (
                        line.split("Global progress:", 1)[1].strip()
                    )

                if "BD pipeline progress:" in line:
                    live_state["stage"] = "Concurrent BD/proxy verification"
                    live_state["processed"] = (
                        line.split("BD pipeline progress:", 1)[1].strip()
                    )
                elif "BD" in line and "progress" in line.lower():
                    live_state["stage"] = "BD protection verification"

        return_code = process.wait()
finally:
    stop_monitor.set()
    monitor_thread.join(timeout=3)

    # Colab-only progress frequency change repository-তে রাখা হবে না।
    settings_path.write_text(
        settings_original_text,
        encoding="utf-8",
    )

elapsed = int(time.monotonic() - started_monotonic)
elapsed_minutes, elapsed_seconds = divmod(elapsed, 60)

print("\n" + "=" * 72)
print("SCANNER PROCESS FINISHED")
print(f"Exit code : {return_code}")
print(f"Duration  : {elapsed_minutes}m {elapsed_seconds}s")
print(f"Full log  : {log_path}")
print("=" * 72)

if return_code != 0:
    print("\n❌ Scanner failed. Last 30 log lines:")
    try:
        log_lines = log_path.read_text(encoding="utf-8").splitlines()
        for log_line in log_lines[-30:]:
            print(log_line)
    except Exception:
        pass

    raise RuntimeError(
        f"Scanner failed with exit code {return_code}. "
        "উপরের exact error দেখুন। কোনো result GitHub-এ push করা হয়নি।"
    )

print("\n[RESULT CHECK] Reading generated output summaries...")

for name, path in tracked_paths.items():
    description = describe_updated_file(name, path)
    if description:
        print(f"✅ {path.relative_to(REPO_DIR)} → {description}")

status_result = subprocess.run(
    ["git", "status", "--short"],
    cwd=REPO_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

if SHOW_CHANGED_FILES:
    print("\n[GIT CHANGES]")
    print(status_result.stdout.rstrip() or "No changed files.")

if PUSH_RESULTS_TO_GITHUB:
    print("\n[PUSH] Staging data/, reports/ and state/...")
    subprocess.run(
        ["git", "add", "-A", "--", "data", "reports", "state"],
        cwd=REPO_DIR,
        check=True,
    )

    diff_result = subprocess.run(
        ["git", "diff", "--cached", "--quiet"],
        cwd=REPO_DIR,
    )

    if diff_result.returncode == 0:
        print("✅ No changed output files. Nothing to commit.")
    else:
        diff_stat = subprocess.run(
            ["git", "diff", "--cached", "--stat"],
            cwd=REPO_DIR,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=True,
        )
        print("\nFiles that will be committed:")
        print(diff_stat.stdout.rstrip())

        timestamp = datetime.now(timezone.utc).strftime(
            "%Y-%m-%d %H:%M:%S UTC"
        )
        commit_message = f"Colab update: {SCAN_MODE} [{timestamp}]"

        subprocess.run(
            ["git", "commit", "-m", commit_message],
            cwd=REPO_DIR,
            check=True,
        )

        print("\n[CLEANUP] Removing non-published scanner leftovers...")

        # working/ contains temporary intermediate scan files. They are not
        # published and must not block pull --rebase after the output commit.
        subprocess.run(
            ["git", "restore", "--worktree", "--", "working"],
            cwd=REPO_DIR,
            check=False,
        )

        # Remove generated Python cache directories/files without touching
        # project source files.
        for cache_dir in REPO_DIR.rglob("__pycache__"):
            if cache_dir.is_dir():
                import shutil
                shutil.rmtree(cache_dir, ignore_errors=True)

        for pyc_file in REPO_DIR.rglob("*.pyc"):
            try:
                pyc_file.unlink()
            except OSError:
                pass

        remaining_status = subprocess.run(
            ["git", "status", "--porcelain"],
            cwd=REPO_DIR,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=True,
        ).stdout.strip()

        if remaining_status:
            print("Remaining non-output changes before rebase:")
            print(remaining_status)
            raise RuntimeError(
                "Repository is not clean after output commit. "
                "Push stopped to avoid overwriting unrelated files."
            )

        print("\n[PUSH] Fetching latest remote branch...")
        run_git(
            ["fetch", "origin", BRANCH],
            cwd=REPO_DIR,
        )

        print("[PUSH] Rebasing local output commit...")
        run_git(
            ["rebase", f"origin/{BRANCH}"],
            cwd=REPO_DIR,
        )

        print("[PUSH] Uploading output commit to GitHub...")
        run_git(
            ["push", "origin", f"HEAD:{BRANCH}"],
            cwd=REPO_DIR,
        )

        pushed_commit = run_git(
            ["rev-parse", "--short", "HEAD"],
            cwd=REPO_DIR,
        ).stdout.strip()

        print("=" * 72)
        print("✅ SCAN AND GITHUB PUSH COMPLETED")
        print(f"   Mode   : {SCAN_MODE}")
        print(f"   Commit : {pushed_commit}")
        print(f"   Branch : {BRANCH}")
        print("=" * 72)
else:
    print("\nGitHub push disabled for this run.")


In [ ]:
#@title 3) Retry last GitHub push only — do not scan again
import shutil
import subprocess

print("[RECOVERY] Looking for an unpushed scan commit...")

run_git(["fetch", "origin", BRANCH], cwd=REPO_DIR)

ahead_result = subprocess.run(
    ["git", "rev-list", "--count", f"origin/{BRANCH}..HEAD"],
    cwd=REPO_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=True,
)
ahead_count = int((ahead_result.stdout or "0").strip() or "0")

if ahead_count < 1:
    raise RuntimeError(
        "কোনো unpushed local scan commit পাওয়া যায়নি। "
        "এই recovery cell চালানোর প্রয়োজন নেই।"
    )

print(f"[RECOVERY] Found {ahead_count} unpushed commit(s).")
print("[RECOVERY] Cleaning temporary non-published files...")

subprocess.run(
    ["git", "restore", "--worktree", "--", "working"],
    cwd=REPO_DIR,
    check=False,
)

for cache_dir in REPO_DIR.rglob("__pycache__"):
    if cache_dir.is_dir():
        shutil.rmtree(cache_dir, ignore_errors=True)

for pyc_file in REPO_DIR.rglob("*.pyc"):
    try:
        pyc_file.unlink()
    except OSError:
        pass

remaining_status = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=REPO_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=True,
).stdout.strip()

if remaining_status:
    print("Remaining changes:")
    print(remaining_status)
    raise RuntimeError(
        "Repository clean হয়নি। কোনো file overwrite না করে recovery থামানো হয়েছে।"
    )

print("[RECOVERY] Rebasing output commit onto latest main...")
run_git(["rebase", f"origin/{BRANCH}"], cwd=REPO_DIR)

print("[RECOVERY] Pushing output commit to GitHub...")
run_git(["push", "origin", f"HEAD:{BRANCH}"], cwd=REPO_DIR)

commit_id = run_git(
    ["rev-parse", "--short", "HEAD"],
    cwd=REPO_DIR,
).stdout.strip()

print("=" * 68)
print("✅ LAST SCAN RESULT PUSHED SUCCESSFULLY")
print(f"Commit: {commit_id}")
print(f"Branch: {BRANCH}")
print("=" * 68)
